In [2]:
from pathlib import Path
import pandas as pd

#cuurent directory in the notebook to reach dataset go up by one level
DATA = Path("../dataset")

#sep="\t" is required, addresses contain commas
s1 = pd.read_csv(DATA / "train/train_source1.tsv",sep="\t")
s2 = pd.read_csv(DATA / "train/train_source2.tsv", sep="\t")
s3 = pd.read_csv(DATA / "train/train_source3.tsv", sep="\t")
gt = pd.read_csv(DATA /"train/train_ground_truth.tsv", sep="\t")

for name, df in [("s1",s1),("s2",s2),("s3",s3),("gt",gt)]:
    print(f"{name} shape: {df.shape}")


s1 shape: (2206821, 4)
s2 shape: (5034616, 4)
s3 shape: (5285603, 4)
gt shape: (2206821, 2)


In [3]:
# country counts show what the data covers; nulls show which fields are unreliable
print(s1["country"].value_counts(normalize=True)*100,"\n")
print(s1.dtypes,"\n")

print(s1.isna().mean(),"\n")
print("S1 entities with no match:", gt["matched_entity_ids"].isna().mean()*100,"\n")
print("Count of empty string in business address ",(s1["business_address"].str.strip() == "").mean())

s1.head()

country
US       59.979174
India    40.020826
Name: proportion, dtype: float64 

entity_id           str
business_name       str
business_address    str
country             str
dtype: object 

entity_id           0.0
business_name       0.0
business_address    0.0
country             0.0
dtype: float64 

S1 entities with no match: 5.584820880352326 

Count of empty string in business address  0.0


,entity_id,business_name,business_address,country
0,S1-925783039,Orelee's Barbershop,"1795 Westchester Drive, High Point, NC",US
1,S1-773889195,Prime Money,"17560 Ellis Road, Tahlequah, OK",US
2,S1-377745466,B+ Retail Inc,"1712 Montebello Avenue, Phoenix, AZ",US
3,S1-133037285,Christ Chapel,"2100 Cameron Drive, Unit APARTMENT G, Dundalk, MD",US
4,S1-755362802,Prabhav Business Center,"797, Lake Town Block A, Kolkata, Howrah, West ...",India


In [4]:
# match-list length shows how many true matches a candidate set must hold per S1 entity
gt["n_matches"] = gt["matched_entity_ids"].fillna("").apply(
    lambda s: len([x for x in s.split(",") if x])
)
print(gt["n_matches"].value_counts().sort_index().head(10))
print("avg matches per S1:", gt["n_matches"].mean())

n_matches
0    123247
1    119157
2    375212
3    530841
4    484115
5    321957
6    164868
7     63968
8     18680
9      4205
Name: count, dtype: int64
avg matches per S1: 3.4612526344456573


In [5]:
# sample first, then filter the 5M-row tables to only the needed IDs (fast, low memory)
matched = gt.dropna(subset=["matched_entity_ids"]).sample(10, random_state=42)

need = {i.strip() for s in matched["matched_entity_ids"] for i in s.split(",") if i.strip()}
s1_rows = s1[s1["entity_id"].isin(matched["source1_entity_id"])].set_index("entity_id")
others = pd.concat([
    s2[s2["entity_id"].isin(need)],
    s3[s3["entity_id"].isin(need)],
]).set_index("entity_id")

for _, row in matched.iterrows():
    r = s1_rows.loc[row["source1_entity_id"]]
    print(f"S1 | {r.business_name} | {r.business_address} | {r.country}")
    for mid in row["matched_entity_ids"].split(","):
        m = others.loc[mid.strip()]
        print(f"   {mid[:2]} | {m.business_name} | {m.business_address} | {m.country}")
    print("-" * 70)

S1 | Davis Family Office | 88 Olive Circle, Lebanon, TN | US
   S2 | Davis Family Offie | 88 OLIVE CIR, LEBANON, TN | US
   S2 | DAVIS FAMILY OFFICE | OLIVE CIR, LEBANON, TN | US
   S3 | Davis Family (Office) | 88 Olive Cir, Lebanon, Tennessee | US
   S3 | Davis Family | Lebanon, Tennessee, 88 Olive Cir | US
----------------------------------------------------------------------
S1 | MS Consultancy Corp | Shymala Appts 1St Floor Flat No. 4 Opp Ratna Hospital Sb Road In Haveli, Pune, Maharashtra | India
   S2 | MS [Consultancy] | SHYMALA APPTS 1-1ST FLOOR FLAT NO. 4 OPP RATNA HOSPITAL SB ROAD IN HAVELI, PUNE, Maharashtra | India
   S3 | M5  Consultancy Corp | Pune, MH, Pune, Shymala Appts 1St Floor Flat No. 4 Opp Ratna Hospital Sb Road In Haveli | India
   S3 | MS Corp Services | Shymala Appts 1St Floor Flat No. 4 Opp Ratna Hospital Sb Road In Haveli, Pune, MH | India
----------------------------------------------------------------------
S1 | 3520 Main Road Realty Inc | TX, 158 Simpson L

In [6]:
# S2/S3 nulls were never checked, and blank strings hide from isna()
for name, df in [("s2", s2), ("s3", s3)]:
    print(name, df.isna().mean().round(4).to_dict())
# whitespace-only cells aren't NaN, so isna() can't see them
for name, df in [("s1", s1), ("s2", s2), ("s3", s3)]:
    for col in ["business_name", "business_address"]:
        print(name, col, (df[col].str.strip() == "").sum())

#  if every matched ID appears once, the matches form a clean one-to-one partition
all_ids = gt["matched_entity_ids"].dropna().str.split(",").explode().str.strip()
print("matched IDs:", len(all_ids), "| unique:", all_ids.nunique())

s2 {'entity_id': 0.0, 'business_name': 0.0, 'business_address': 0.0336, 'country': 0.0}
s3 {'entity_id': 0.0, 'business_name': 0.0, 'business_address': 0.0333, 'country': 0.0}
s1 business_name 0
s1 business_address 0
s2 business_name 0
s2 business_address 0
s3 business_name 0
s3 business_address 0
matched IDs: 7638365 | unique: 7638365


In [7]:
import gc
from sklearn.model_selection import train_test_split

N_DEV = 100_000

# a random S1 sample keeps the real match-count mix and singleton rate
gt_dev = gt[["source1_entity_id", "matched_entity_ids"]].sample(N_DEV, random_state=42)

# every true match of a sampled S1 entity must be in the pool, or recall is understated
dev_match_ids = set(
    gt_dev["matched_entity_ids"].dropna().str.split(",").explode().str.strip()
)

#  distractors are sampled at the same rate as S1, so the pool keeps the real
# ratio of unmatched records to matched ones
frac = N_DEV / len(gt)

def build_pool(df):
    matches = df[df["entity_id"].isin(dev_match_ids)]
    unmatched = df[~df["entity_id"].isin(all_ids)]
    return pd.concat([matches, unmatched.sample(frac=frac, random_state=42)]).copy()

# free each full table right after use, since RAM is the tight resource
s1_dev = s1[s1["entity_id"].isin(gt_dev["source1_entity_id"])].copy()
del s1
s2_dev = build_pool(s2); del s2; gc.collect()
s3_dev = build_pool(s3); del s3; gc.collect()
del all_ids; gc.collect()

# : split by S1 entity so an entity and its matches stay on the same side
train_ids, val_ids = train_test_split(
    gt_dev["source1_entity_id"], test_size=0.2, random_state=42
)

pool_ids = set(s2_dev["entity_id"]) | set(s3_dev["entity_id"])
print("all true matches in pool:", dev_match_ids <= pool_ids)
print(len(s1_dev), len(s2_dev), len(s3_dev), len(train_ids), len(val_ids))

all true matches in pool: True
100000 228117 239377 80000 20000


In [10]:
out = DATA / "dev"
out.mkdir(exist_ok=True)
for name, df in [("s1", s1_dev), ("s2", s2_dev), ("s3", s3_dev), ("gt", gt_dev)]:
    df.to_csv(out / f"{name}_dev.tsv", sep="\t", index=False)
train_ids.to_csv(out / "train_ids.txt", index=False, header=False)
val_ids.to_csv(out / "val_ids.txt", index=False, header=False)

In [11]:
import re

# count non-Latin text to decide if a state-name mapping table is worth building
nonlatin = re.compile(r"[^\x00-\u024F]")
for name, df in [("s1", s1_dev), ("s2", s2_dev), ("s3", s3_dev)]:
    for col in ["business_name", "business_address"]:
        pct = df[col].fillna("").str.contains(nonlatin).mean() * 100
        print(name, col, round(pct, 2), "%")

s1 business_name 0.0 %
s1 business_address 0.0 %
s2 business_name 9.45 %
s2 business_address 9.54 %
s3 business_name 5.23 %
s3 business_address 9.14 %


In [12]:
import unicodedata
from collections import Counter

def script_counts(series, top=8):
    #show which writing systems (or symbols) trigger the flag before designing a fix
    flagged = series.fillna("")[series.fillna("").str.contains(nonlatin)]
    c = Counter()
    for s in flagged:
        for ch in s:
            if ord(ch) > 0x24F:
                c[unicodedata.name(ch, "UNKNOWN").split()[0]] += 1
    return c.most_common(top)

for name, df in [("s2", s2_dev), ("s3", s3_dev)]:
    for col in ["business_name", "business_address"]:
        print(name, col, script_counts(df[col]))

#  is it only one country, and what do real examples look like
mask = s2_dev["business_name"].str.contains(nonlatin, na=False)
print(s2_dev[mask]["country"].value_counts())
print(s2_dev[mask][["business_name", "business_address"]].head(8).to_string())

s2 business_name [('DEVANAGARI', 314795), ('TELUGU', 52067), ('KANNADA', 51968), ('TAMIL', 49505), ('BENGALI', 40093), ('GUJARATI', 36108), ('MALAYALAM', 25971), ('ORIYA', 11468)]
s2 business_address [('DEVANAGARI', 109102), ('BENGALI', 14550), ('TAMIL', 13887), ('TELUGU', 13425), ('KANNADA', 12285), ('GUJARATI', 8826), ('MALAYALAM', 3690), ('ORIYA', 1560)]
s3 business_name [('DEVANAGARI', 173804), ('TAMIL', 29082), ('TELUGU', 28744), ('KANNADA', 28639), ('BENGALI', 21387), ('GUJARATI', 20426), ('MALAYALAM', 15016), ('ORIYA', 6365)]
s3 business_address [('DEVANAGARI', 109179), ('BENGALI', 14860), ('TAMIL', 14067), ('TELUGU', 13439), ('KANNADA', 12432), ('GUJARATI', 8760), ('MALAYALAM', 3545), ('GURMUKHI', 1580)]
country
India    21555
Name: count, dtype: int64
                              business_name                                                                                       business_address
91              फर्स्ट फूड प्राइवेट लिमिटेड                                       

In [14]:
import re
import unicodedata

SUFFIXES = {"pvt", "private", "ltd", "limited", "llc", "inc", "corp",
            "corporation", "llp", "pllc", "co"}

def norm_name(s):
    #  strip case, accents, punctuation and legal suffixes, the easy noise seen in the pairs
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return " ".join(w for w in s.split() if w not in SUFFIXES)

pool = pd.concat([s2_dev, s3_dev], ignore_index=True)
pool["name_norm"] = pool["business_name"].map(norm_name)

#: native-script names become "" after cleaning and would all collide on one key,
# so they are left out. Their true matches count as misses, which is the honest floor
pool = pool[pool["name_norm"] != ""].copy()

# country is part of the key so US and India never compete (no hard-coded country list)
pool["key"] = pool["country"] + "|" + pool["name_norm"]
s1_dev["key"] = s1_dev["country"] + "|" + s1_dev["business_name"].map(norm_name)

groups = pool.groupby("key")["entity_id"].apply(list).to_dict()

truth = {
    r.source1_entity_id: ({i.strip() for i in r.matched_entity_ids.split(",")}
                          if isinstance(r.matched_entity_ids, str) else set())
    for r in gt_dev.itertuples()
}

def f_half(pred, true):
    # a singleton scores 1.0 only if we predict nothing, and 0.0 for any prediction
    if not true:
        return 1.0 if not pred else 0.0
    tp = len(pred & true)
    if tp == 0:
        return 0.0
    p, r = tp / len(pred), tp / len(true)
    return 1.25 * p * r / (0.25 * p + r)

val_set = set(val_ids)
rows = []
for r in s1_dev[s1_dev["entity_id"].isin(val_set)].itertuples():
    pred = set(groups.get(r.key, []))
    rows.append((r.entity_id, len(truth[r.entity_id]), len(pred),
                 f_half(pred, truth[r.entity_id])))

res = pd.DataFrame(rows, columns=["s1", "n_true", "n_pred", "f05"])
print("F0.5 (macro):", round(res["f05"].mean(), 4))
print("singletons:", round(res[res.n_true == 0]["f05"].mean(), 4),
      "| with matches:", round(res[res.n_true > 0]["f05"].mean(), 4))
print("avg predictions per S1:", round(res["n_pred"].mean(), 2))

F0.5 (macro): 0.6028
singletons: 0.7603 | with matches: 0.5935
avg predictions per S1: 3.29


In [15]:
tp_total = pred_total = true_total = 0
singleton_false = n_singletons = 0

for r in s1_dev[s1_dev["entity_id"].isin(val_set)].itertuples():
    true = truth[r.entity_id]
    pred = set(groups.get(r.key, []))

    if not true:
        # WHY: singleton predictions are pure false matches, tracked apart from the match-level counts
        n_singletons += 1
        singleton_false += bool(pred)
        continue

    tp_total += len(pred & true)
    pred_total += len(pred)
    true_total += len(true)

# WHY: "micro" pools all counts first, so entities with many matches weigh more than in the macro F0.5
print("precision (entities with matches):", round(tp_total / pred_total, 4))
print("recall    (entities with matches):", round(tp_total / true_total, 4))
print("singletons falsely matched:", singleton_false, "of", n_singletons)

precision (entities with matches): 0.4998
recall    (entities with matches): 0.463
singletons falsely matched: 267 of 1114
